# Alerta Mensal de Evasão — Graduação Híbrida
### a instituicao | v3 — três modelos, um por corte do trimestre

O modelo não existe para acertar uma métrica. Ele existe para **ordenar uma
fila**: quem a equipe contata primeiro.

## O que mudou na base (02/10/2026)

A query `03_extracao_hibrido.sql` recebeu o mesmo conserto aplicado ao EAD no dia
anterior: **`cursou_alvo` passou a ler `ERP_MATRICULA` além de `ERP_HISTMATRICULA`**.

Período que acaba de fechar ainda não migrou para o histórico. No 2026/23, alvo da
coorte nova, havia **8 alunos no histórico contra 3.267 em `ERP_MATRICULA`**. Lendo
só o histórico, quase toda a coorte sairia rotulada como evasora — e o erro é
silencioso: a query não reclama, só devolve uma coluna de 1.

Tem que ser `UNION`, nunca troca: a migração **apaga** de `ERP_MATRICULA`, então
somando as duas o aluno é encontrado em qualquer fase.

Os defeitos estruturais do híbrido (`INNER JOIN` no AVA, `EVADIU` sem o teste de
`so`, `ELEGIVEL` com view sem data) já haviam sido corrigidos na auditoria de
**23/09/2026**.

## Duas diferenças em relação ao caderno do EAD

**1. `CURSO` é normalizado pelos três últimos dígitos.** Seis cursos do híbrido
têm dois códigos (`2111`/`8111`, `2256`/`8256`...) — são 38,5% da base. O `8xxx` é
o credenciamento novo do mesmo curso, e é onde os calouros de 2026 entram. Sem
normalizar, o modelo aprende `CURSO_2111` e não reconhece ninguém quando a turma
nova chegar toda no `8111`. O agrupamento reduz 29 códigos a 23.

**2. Não há bloco financeiro.** As 9 colunas `FIN_*` existem no EAD e no
presencial, não aqui. O modelo do híbrido enxerga só comportamento acadêmico e de
AVA. Portar o bloco é possível e está em aberto — no EAD ele piorou o topo da fila
(−4,5 no topo 200) e melhorou o corpo (+39 no `recall@10%`).

## Por que três modelos e não um

A informação sobre o aluno cresce ao longo do trimestre. Aos 30 dias mal existe
comportamento no AVA; no fechamento existe o trimestre inteiro. Um modelo só não
atende os dois momentos.

## As coortes, e por que a 2025/23 não serve de teste

| Coorte | Alunos | Evasores | Taxa | Sem AVA | m1 (+30d) | m2 (+60d) | m3 (fechamento) |
|---|---|---|---|---|---|---|---|
| 2025/23 | 2.421 | 271 | 11,2% | 7,0% | 31/07/25 | 30/08/25 | 01/10/25 |
| 2025/24 | 2.236 | 345 | 15,4% | 8,3% | 31/10/25 | 30/11/25 | 30/12/25 |
| 2026/21 | 2.715 | 252 | 9,3% | 5,1% | 31/01/26 | 02/03/26 | 01/05/26 |
| **2026/22** | **2.744** | **372** | **13,6%** | 8,2% | 31/05/26 | 30/06/26 | 30/07/26 |

O corte `m1` da 2025/23 cai em **31/07/2025**, mês de rampa do AVA. Ela fica no
**treino**; medir nela não diz nada.

**A quarta coorte entrou em 02/10/2026**, quando o trimestre 2026/23 fechou e o
alvo dela ficou pronto. Com ela existem **duas origens deslizantes** sem depender
da coorte contaminada — é a primeira vez que dá para saber se o resultado se
repete ou foi sorte de uma turma. O tópico 16 mede as duas.

> **O calendário do híbrido é irregular.** O `21` dura 120 dias e os outros 90, e
> o 2026/23 durou 66. Os cortes `m1` e `m2` são sempre +30 e +60 dias do início,
> então são comparáveis entre coortes; o `m3` é sempre o fechamento, e a distância
> entre `m2` e `m3` é que varia.

> **A evasão oscila entre 9,3% e 15,4%** sem tendência. Como a taxa muda, não
> compare o número bruto de evasores capturados entre coortes — use o ganho sobre
> o acaso, no tópico 16.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)

SEMENTE = 42
SEMENTES = range(5)
CORTES = [1, 2, 3]
# 4 coortes a partir de 01/10/2026 -> 2 origens deslizantes.
# A principal e a mais recente: treina em tudo que veio antes.
TREINO = ["2025/23", "2025/24", "2026/21"]
TESTE = "2026/22"

LIMIAR = 0.30       # mesmo limiar do presencial, para as tabelas serem comparaveis
FILA = 200          # mesma fila de referencia do presencial

# faixas da operacao: so CRITICA e ALTA recebem contato
PCT_ALTA, PCT_MEDIA, PCT_CRITICA = 0.10, 0.20, 0.20

RAIZ = Path.cwd() if False else None

## 1. Carregamento dos Dados

In [ ]:
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "tabelas").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent

DTYPE = {"ALUNO": str, "CURSO": str, "END_MUNICIPIO": str}
bruto = {m: pd.read_csv(RAIZ / "tabelas" / f"extracao_hibrido_m{m}.csv",
                        encoding="utf-8-sig", low_memory=False, dtype=DTYPE)
         for m in CORTES}

resumo = (bruto[1].groupby("COORTE")
          .agg(alunos=("EVADIU", "size"), evasores=("EVADIU", "sum")))
resumo["taxa"] = (100 * resumo.evasores / resumo.alunos).round(1)
for m in CORTES:
    resumo[f"corte_m{m}"] = bruto[m].groupby("COORTE").DT_CORTE.first()

print(f"{bruto[1].shape[1]} colunas por corte")
print()
print(resumo.to_string())

## 2. Elegibilidade — o filtro virou no-op (28/09/2026)

`ELEGIVEL` agora é `CAST(1 AS int)` na query. O filtro abaixo não descarta
ninguém e fica por compatibilidade.

**Por quê:** a fórmula antiga decidia quem entra na base usando
`VW_ALUNO_SITUACAO_PERIODO`, que guarda um estado por período **sem data**. Não
há como fatiar por `DT_CORTE`, então ela descartava alunos usando informação que
só existiria meses depois.

Não era métrica inflada — era base **irreproduzível em produção**. No dia do
corte ninguém sabe quem vai concluir no fim do trimestre.

> **Rótulo pode usar o futuro; população não.** O rótulo *é* o que aconteceu
> depois. Quem entra na base tem que ser decidido com o que existe em `DT_CORTE`.

A auditoria de quem concluiu ou transferiu continua em `AUDITORIA_STATUS`.

In [ ]:
dados = {m: bruto[m][bruto[m].ELEGIVEL == 1].reset_index(drop=True) for m in CORTES}

for m in CORTES:
    d = dados[m]
    desc = len(bruto[m]) - len(d)
    print(f"m{m}: {len(d):,} elegiveis de {len(bruto[m]):,}  "
          f"(descartados: {desc})")
print()
print("AUDITORIA_STATUS no corte de fechamento:")
print(dados[3].AUDITORIA_STATUS.value_counts().to_string())

## 3. Engenharia de Features de Risco

Quatro transformações, cada uma com um motivo medido.

**Rank do AVA dentro da coorte.** A cobertura do log varia entre trimestres —
71%, 22% e 47% de alunos sem dado no `m1` das três coortes. Um modelo treinado
num trimestre e aplicado em outro lê número absoluto errado; o rank percentual
dentro da própria coorte resolve.

**Razões.** `MAIOR_GAP / DIAS_ATIVOS` e `SESSOES / DIAS_ATIVOS` capturam
intensidade, que nenhuma das colunas isoladas mostra.

**`SEM_AVA`.** Não ter registro nenhum no AVA é sinal, não ausência de dado — e
foi exatamente o que o `INNER JOIN` antigo apagava.

**Cauda longa em `OUTRO`.** `CURSO` e `END_MUNICIPIO` são códigos, não números.
Fora dos 25 mais comuns, tudo vira `OUTRO`.

In [ ]:
AVA = [c for c in dados[1].columns if c.startswith("AVA_")]


def engenharia(d):
    d = d.copy()
    for c in AVA:
        d[c + "_RANK"] = d.groupby("COORTE")[c].rank(pct=True)
    d["RATIO_GAP_ATIVOS"] = d.AVA_MAIOR_GAP_DIAS / (d.AVA_DIAS_ATIVOS + 1)
    d["SESSOES_POR_DIA"] = d.AVA_QTD_SESSOES / (d.AVA_DIAS_ATIVOS + 1)
    d["SEM_AVA"] = d[AVA].isna().all(axis=1).astype(int)
    # CURSO duplicado: seis cursos do hibrido tem dois codigos (2111/8111 ...),
    # sempre com os tres ultimos digitos iguais. O 8xxx e o credenciamento novo
    # do MESMO curso -- e onde os calouros de 2026 entram. Sem normalizar, o
    # modelo aprende CURSO_2111 e nao reconhece ninguem quando a turma nova
    # chegar toda no 8111. Sao 38,5% da base. Ver CLAUDE.md.
    d["CURSO"] = d.CURSO.astype(str).str.strip().str[-3:]
    for c in ("CURSO", "END_MUNICIPIO"):
        top = d[c].value_counts().head(25).index
        d[c] = np.where(d[c].isin(top), d[c].astype(str), "OUTRO")
    return d


feat = {m: engenharia(dados[m]) for m in CORTES}
print(f"{len(AVA)} colunas de AVA")
print(f"{feat[1].shape[1] - dados[1].shape[1]} colunas criadas")

## 4. Separação de $X$ (Features) e $y$ (Target)

| Coluna | Por que sai |
|---|---|
| `CURSOU_ALVO` | **é** o rótulo — vazamento puro |
| `SIT_ALVO`, `SIT_FIM_OBS`, `AUDITORIA_STATUS` | derivam do alvo |
| `ALUNO` | identificador — o modelo decora o aluno |
| `DT_CORTE`, `DT_INICIO_OBS`, `MES_CORTE` | auditoria |
| `COORTE`, `ANO_OBS`, `PERIODO_OBS` | o modelo decoraria o trimestre |
| `SEXO` | **ressalva ética** — usar como preditor direciona intervenção por gênero |
| `NOME_CURSO` | redundante com `CURSO` |

`SEXO` fica guardado à parte, só para a auditoria de viés.

### O bloco de AVA entra ou sai? Aqui a pergunta e **medida**, nao herdada

No híbrido, remover todas as colunas de AVA dos cortes precoces rendeu +41 e +74
evasores. A causa era a cobertura do log variar demais entre trimestres: o modelo
aprendia "pouco acesso = risco", mas na turma nova pouco acesso as vezes e o aluno
sumindo e as vezes e o log ainda nao ter registrado.

**No hibrido a premissa e mais fraca**, porque a cobertura do AVA e muito melhor:

| Modalidade | Alunos sem nenhum dado de AVA |
|---|---|
| híbrido | ~43% |
| **Hibrido** | **5% a 8%** |

Faz sentido -- aqui parte da aula e presencial, entao o aluno existe no sistema
mesmo sem entrar no ambiente virtual. Se a cobertura e estavel, a confusao que
justificou a remocao no híbrido nao deveria aparecer.

**Entao a celula abaixo mede, em vez de presumir.** A decisao sai de uma origem
interna (treina 2025/23, valida 2025/24) e e aplicada sem troca nas origens de
teste -- do contrario seria escolha no conjunto de teste.


In [ ]:
colunas_remover = ["ALUNO", "EVADIU", "CURSOU_ALVO", "SIT_ALVO", "ELEGIVEL",
                   "SIT_FIM_OBS", "AUDITORIA_STATUS", "DT_CORTE", "DT_INICIO_OBS",
                   "COORTE", "ANO_OBS", "PERIODO_OBS", "SEXO", "NOME_CURSO",
                   "MES_CORTE"]

BLOCO_AVA = ([c for c in feat[1].columns if c.startswith("AVA_")]
             + ["RATIO_GAP_ATIVOS", "SESSOES_POR_DIA", "SEM_AVA"])


def montar_X(m, fora):
    base = feat[m].drop(columns=[c for c in fora if c in feat[m].columns]).copy()
    cat = base.select_dtypes(include=["object"]).columns.tolist()
    for c in cat:
        base[c] = base[c].fillna("NAO_INFORMADO").astype(str)
    return pd.get_dummies(base, columns=cat)


def medir(m, fora, treino, teste, sementes=range(5)):
    """Evasores alcancados na fila de 10%, media das sementes."""
    Xm = montar_X(m, fora)
    ym, co = feat[m].EVADIU.astype(int).values, feat[m].COORTE.values
    tr, te = np.isin(co, treino), co == teste
    k = int(np.ceil(PCT_ALTA * te.sum()))
    r = []
    for s in sementes:
        mod = RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                                     class_weight="balanced", n_jobs=-1,
                                     random_state=s)
        mod.fit(Xm[tr], ym[tr])
        p = mod.predict_proba(Xm[te])[:, 1]
        r.append(ym[te][np.argsort(-p)[:k]].sum())
    return float(np.mean(r)), k


# A decisao sai de uma origem INTERNA -- nunca das coortes de teste.
VAL_TREINO, VAL_TESTE = ["2025/23"], "2025/24"
REMOVER_POR_CORTE, linhas_ava = {}, []
for m in CORTES:
    com, k = medir(m, colunas_remover, VAL_TREINO, VAL_TESTE)
    sem, _ = medir(m, colunas_remover + BLOCO_AVA, VAL_TREINO, VAL_TESTE)
    tirar = sem > com
    REMOVER_POR_CORTE[m] = BLOCO_AVA if tirar else []
    linhas_ava.append({"Corte": f"m{m}", "Fila": k, "Com AVA": round(com, 1),
                       "Sem AVA": round(sem, 1), "Diferenca": round(sem - com, 1),
                       "Escolhido": "sem AVA" if tirar else "com AVA"})

display(pd.DataFrame(linhas_ava))
print(f"Decidido na origem interna: treina {VAL_TREINO}, valida em {VAL_TESTE}.")
print("A escolha vale para as duas origens de teste, sem nova medicao.")
print()

X, y, meta = {}, {}, {}
for m in CORTES:
    d = feat[m]
    X[m] = d.drop(columns=[c for c in colunas_remover + REMOVER_POR_CORTE[m]
                           if c in d.columns])
    y[m] = d.EVADIU.astype(int).values
    meta[m] = {"coorte": d.COORTE.values, "sexo": d.SEXO.values,
               "aluno": d.ALUNO.values}

for m in CORTES:
    tem = sum(1 for c in X[m].columns if c.startswith("AVA_"))
    print(f"m{m}: {X[m].shape[1]:>2} features  |  colunas de AVA: {tem}")


## 5. Encoding de Categóricas e Valores Ausentes

**Nulo não é imputado.** Em `AVA_*` o nulo significa "não há registro de
acesso", que é quase o próprio sinal de risco — e o RandomForest do sklearn ≥ 1.4
trata NaN nativamente. Imputar a média apagaria a informação.

Em coluna **categórica** o nulo vira categoria própria (`NAO_INFORMADO`) — sem
isso a linha sairia com zero em todas as colunas codificadas, indistinguível da
categoria de referência e calada.

In [ ]:
X_enc, agrupamentos = {}, {}
for m in CORTES:
    base = X[m].copy()
    cat = base.select_dtypes(include=["object"]).columns.tolist()
    for c in cat:
        base[c] = base[c].fillna("NAO_INFORMADO").astype(str)
    X_enc[m] = pd.get_dummies(base, columns=cat)
    agrupamentos[m] = {c: sorted(base[c].unique().tolist()) for c in cat}

print(f"X com {X_enc[1].shape[1]} colunas apos one-hot")
for c, v in agrupamentos[1].items():
    print(f"   {c}: {len(v)} categorias")
print()
nulos = X[1].isna().mean().sort_values(ascending=False).head(6)
print("colunas com mais nulo (nao imputados):")
print((100 * nulos).round(1).to_string())

## 6. Validação: Origem Deslizante (em vez de CV out-of-fold)

**Treina no passado, testa na turma nova.** É assim que o modelo vai ser usado.

O híbrido tem **quatro coortes** desde 01/10/2026, o que permite origem deslizante de
verdade — coisa que o presencial não tem, com uma coorte só. Com a 2025/23 fora da
avaliação, sobram **duas origens**:

```
origem 1   treina 2025/23 + 2025/24              testa 2026/21
origem 2   treina 2025/23 + 2025/24 + 2026/21    testa 2026/22
```

Split aleatório não entra: ele infla a métrica num problema temporal.

**Os tópicos daqui até o 15 usam a origem 2** — a mais recente, treinada em tudo
que veio antes, que é o que produção faria. O **tópico 16 mede as duas** e é lá
que se responde se o resultado se repete ou foi sorte de coorte.

In [ ]:
def treinar_e_pontuar(m, sementes=SEMENTES):
    """Treina nas coortes de TREINO e pontua a de TESTE. Media das sementes."""
    Xm, ym, co = X_enc[m], y[m], meta[m]["coorte"]
    tr, te = np.isin(co, TREINO), co == TESTE
    ps, mods = [], []
    for s in sementes:
        mod = RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                                     class_weight="balanced", n_jobs=-1,
                                     random_state=s)
        mod.fit(Xm[tr], ym[tr])
        ps.append(mod.predict_proba(Xm[te])[:, 1])
        mods.append(mod)
    return np.mean(ps, axis=0), mods[0], tr, te


def faixas(n):
    lim_alta = int(np.ceil(PCT_ALTA * n))
    return {"CRITICA": int(np.ceil(PCT_CRITICA * lim_alta)),
            "ALTA": lim_alta,
            "MEDIA": int(np.ceil(PCT_MEDIA * n))}


print(f"treino: {TREINO}")
print(f"teste : {TESTE}")
print(f"sementes: {len(list(SEMENTES))}")

## 7. Treinamento dos Três Modelos (Random Forest Sensível a Custo)

In [ ]:
modelos, proba, y_av, feat_av = {}, {}, {}, {}
for m in CORTES:
    proba[m], modelos[m], tr, te = treinar_e_pontuar(m)
    y_av[m] = y[m][te]
    feat_av[m] = feat[m][te].reset_index(drop=True)

CORTE_DATA = {m: str(feat_av[m].DT_CORTE.iloc[0])[:10] for m in CORTES}

print(f"{'Corte':>6s} {'Data':>12s} {'Alunos':>8s} {'Evasores':>9s} "
      f"{'Arvores':>8s} {'Colunas':>8s} {'Peso classe':>12s}")
print("-" * 70)
for m in CORTES:
    mod = modelos[m]
    print(f"{'m' + str(m):>6s} {CORTE_DATA[m]:>12s} {len(y_av[m]):8,} "
          f"{int(y_av[m].sum()):9d} {mod.n_estimators:8d} "
          f"{X_enc[m].shape[1]:8d} {'balanced':>12s}")

## 8. O Limiar de Decisão e o Tamanho da Fila

O modelo **não é calibrado** (`class_weight='balanced'`), então a probabilidade
serve para ordenar, não para ler como percentual de chance.

A tabela abaixo mostra o problema: o limiar 0,30 marca quase metade da base. É
por isso que a operação usa **posição na fila**, não limiar.

In [ ]:
linhas = []
for m in CORTES:
    for t in (0.50, 0.30, 0.20):
        pred = (proba[m] >= t).astype(int)
        linhas.append({
            "Corte": f"m{m}", "Data": CORTE_DATA[m], "Limiar": f"{t:.2f}",
            "Recall": f"{recall_score(y_av[m], pred, zero_division=0):.1%}",
            "Precisão": f"{precision_score(y_av[m], pred, zero_division=0):.1%}",
            "Alertas": int(pred.sum()),
            "% da base": f"{pred.sum()/len(y_av[m]):.0%}",
        })
display(pd.DataFrame(linhas))

lim = faixas(len(y_av[1]))
print("As faixas que a operacao usa, por POSICAO na fila:")
print(f"   CRITICA  : {lim['CRITICA']:5,} alunos (2% da base)   -> ligacao")
print(f"   ALTA     : {lim['ALTA']:5,} alunos (10% da base)  -> mensagem")
print(f"   MEDIA    : {lim['MEDIA']:5,} alunos (20% da base)  -> so registro")

## 9. Avaliação: quantos evasores a fila alcança

Mesmas colunas da tabela do presencial, para os dois cadernos serem lidos lado a
lado.

In [ ]:
def metricas(ym, p):
    pred = (p >= LIMIAR).astype(int)
    o = np.argsort(-p)
    return {
        "recall@200": ym[o[:FILA]].sum() / ym.sum(),
        "topo200": int(ym[o[:FILA]].sum()),
        "prec@200": ym[o[:FILA]].sum() / FILA,
        "recall@10%": ym[o[:int(0.10 * len(ym))]].sum() / ym.sum(),
        "pr_auc": average_precision_score(ym, p),
        "roc_auc": roc_auc_score(ym, p),
        "recall_limiar": recall_score(ym, pred, zero_division=0),
        "prec_limiar": precision_score(ym, pred, zero_division=0),
        "alertas": int(pred.sum()),
    }


res = {m: metricas(y_av[m], proba[m]) for m in CORTES}
tab = pd.DataFrame([{
    "Corte": f"m{m}", "Data": CORTE_DATA[m], "Evasores": int(y_av[m].sum()),
    "Topo 200 pega": res[m]["topo200"],
    "recall@200": f"{res[m]['recall@200']:.1%}",
    "prec@200": f"{res[m]['prec@200']:.1%}",
    "recall@10%": f"{res[m]['recall@10%']:.1%}",
    "PR-AUC": f"{res[m]['pr_auc']:.3f}",
    "ROC-AUC": f"{res[m]['roc_auc']:.3f}",
    "Recall @0,30": f"{res[m]['recall_limiar']:.1%}",
    "Alertas": res[m]["alertas"],
} for m in CORTES])
display(tab)

n, ev = len(y_av[1]), int(y_av[1].sum())
print(f"teste: {TESTE} -- {n:,} alunos, {ev:,} evasores ({ev/n:.1%})")
print(f"acaso no topo {FILA}: {round(FILA*ev/n)} evasores")
print()
print("Leitura: o topo 200 CAI ao longo do trimestre "
      f"({res[1]['topo200']} -> {res[3]['topo200']}) enquanto o recall@10% SOBE "
      f"({res[1]['recall@10%']:.0%} -> {res[3]['recall@10%']:.0%}).")
print("Sao coisas diferentes. No m1 o topo da fila e quem ja abandonou -- denso e")
print("facil. No m3 o modelo espalha acerto por toda a fila, que e o que a")
print("operacao usa.")

## 10. A Política de Fila: por que não repetir importa

O mesmo aluno tende a aparecer no topo dos três cortes. Se a equipe contatar
sempre os primeiros da lista, gasta três contatos com a mesma pessoa e deixa de
alcançar outras.

A política **"sem repetir"** exclui quem já foi contatado nos cortes anteriores.

In [ ]:
por_corte = {m: pd.DataFrame({"aluno": feat_av[m].ALUNO.astype(str).values,
                              "p": proba[m], "evadiu": y_av[m]})
             for m in CORTES}
TOTAL_EVASORES = int(y_av[3].sum())

resultados = {}
for politica in ("Ingênua", "Sem repetir"):
    contatados, alcancados, hist = set(), set(), []
    for m in CORTES:
        d = por_corte[m].sort_values("p", ascending=False)
        if politica == "Sem repetir":
            d = d[~d["aluno"].isin(contatados)]
        fila = d.head(FILA)
        repetidos = len(set(fila["aluno"]) & contatados)
        alcancados |= set(fila.loc[fila["evadiu"] == 1, "aluno"])
        contatados |= set(fila["aluno"])
        hist.append({"Política": politica, "Corte": f"m{m}", "Data": CORTE_DATA[m],
                     "Evasores na fila": int(fila["evadiu"].sum()),
                     "Já contatados antes": repetidos,
                     "Evasores alcançados": len(alcancados),
                     "% da evasão": f"{len(alcancados)/TOTAL_EVASORES:.1%}"})
    resultados[politica] = (hist, len(contatados), len(alcancados))

display(pd.DataFrame([h for p in resultados for h in resultados[p][0]]))

print(f"Ao fim dos 3 cortes, com {len(CORTES)*FILA:,} contatos em ambos os casos:")
for p, (_, n_ct, n_al) in resultados.items():
    print(f"   {p:12s}: {n_ct:5,} alunos distintos | "
          f"{n_al:4d} evasores alcancados de {TOTAL_EVASORES} "
          f"({n_al/TOTAL_EVASORES:.1%})")

## 11. Curvas: captura acumulada e Precision-Recall por corte

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for pol, cor in [("Ingênua", "#A0AEC0"), ("Sem repetir", "#1A365D")]:
    h = resultados[pol][0]
    axes[0].plot([x["Corte"] for x in h], [x["Evasores alcançados"] for x in h],
                 "o-", lw=2.6, ms=7, color=cor, label=pol)
axes[0].set_title("Evasores alcançados ao longo do trimestre" + chr(10)
                  + f"({FILA} contatos por corte nos dois casos)", fontweight="bold")
axes[0].set_xlabel("Corte do trimestre", fontweight="bold")
axes[0].set_ylabel("Evasores distintos alcançados", fontweight="bold")
axes[0].legend()
axes[0].grid(alpha=0.3)

cores = plt.cm.viridis(np.linspace(0.15, 0.85, len(CORTES)))
for m, cor in zip(CORTES, cores):
    pr, rc, _ = precision_recall_curve(y_av[m], proba[m])
    axes[1].plot(rc, pr, lw=2.2, color=cor,
                 label=f"m{m} ({CORTE_DATA[m]}) — PR-AUC {res[m]['pr_auc']:.3f}")
axes[1].axhline(y_av[3].mean(), color="crimson", ls="--", lw=1.4,
                label=f"Baseline ({y_av[3].mean():.1%})")
axes[1].set_title("Curva Precision-Recall por corte", fontweight="bold")
axes[1].set_xlabel("Recall", fontweight="bold")
axes[1].set_ylabel("Precisão", fontweight="bold")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Importância das Features — como o sinal muda ao longo do trimestre

In [ ]:
imp = pd.DataFrame({f"m{m}": pd.Series(modelos[m].feature_importances_,
                                       index=X_enc[m].columns)
                    for m in CORTES}).fillna(0)
ordem = imp["m3"].sort_values(ascending=False).head(16).index[::-1]
pos = np.arange(len(ordem))

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(pos + 0.20, imp.loc[ordem, "m1"], 0.38, color="#A0AEC0",
        edgecolor="black", lw=0.4, label=f"m1 ({CORTE_DATA[1]}) — 30 dias")
ax.barh(pos - 0.20, imp.loc[ordem, "m3"], 0.38, color="#1A365D",
        edgecolor="black", lw=0.4, label=f"m3 ({CORTE_DATA[3]}) — fechamento")
ax.set_yticks(pos)
ax.set_yticklabels(ordem, fontsize=8.5)
ax.set_xlabel("Importância no Random Forest", fontsize=10, fontweight="bold")
ax.set_title("O que o modelo usa no início e no fim do trimestre",
             fontsize=12, fontweight="bold", pad=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

for m in CORTES:
    print(f"--- m{m} | top 8 ---")
    print(imp[f"m{m}"].sort_values(ascending=False).head(8).round(4).to_string())
    print()

## 13. Salvamento do Artefato (`.pkl`)

**Regra para qualquer artefato novo** (nasceu de um defeito real na Pós Live,
onde o `.pkl` gravava a lista de features *antes* do one-hot e quebrava em
produção):

- `colunas_modelo` — o que o `predict` aceita, depois do one-hot
- `features_pre_encoding` — só para auditoria
- `agrupamentos` — as categorias usadas no one-hot
- um `assert` comparando com `n_features_in_` do modelo

In [ ]:
artefato = {
    "versao": "hibrido_mensal_2026_22_recesso",
    "modelos": modelos,
    "colunas_modelo": {m: X_enc[m].columns.tolist() for m in CORTES},
    "features_pre_encoding": {m: X[m].columns.tolist() for m in CORTES},
    "agrupamentos": agrupamentos,
    "cortes": CORTE_DATA,
    "threshold": LIMIAR,
    "fila": FILA,
    "faixas": {"PCT_CRITICA_DA_ALTA": PCT_CRITICA, "PCT_ALTA": PCT_ALTA,
               "PCT_MEDIA": PCT_MEDIA},
    "treino": TREINO,
    "teste": TESTE,
    "proba_oof": {m: dict(zip(feat_av[m].ALUNO.astype(str), proba[m])) for m in CORTES},
    "metadata": {"por_corte": res},
    "politica_fila": "so CRITICA e ALTA recebem contato; MEDIA e gravada para historico",
    "observacoes": (
        "Faixas por POSICAO na fila, nunca por probabilidade -- o modelo nao e "
        "calibrado. A coorte 2025/23 fica fora da avaliacao (rampa do AVA). "
        "O m2 NAO passa no criterio de publicacao -- ver topico 15."
    ),
}
for m in CORTES:
    assert len(artefato["colunas_modelo"][m]) == modelos[m].n_features_in_, m

destino = RAIZ / "ml" / "grad-hib" / "modelo_evasao_hibrido_mensal.pkl"
joblib.dump(artefato, destino, compress=3)   # sem compress passa de 100 MB no GitHub
print(f"salvo em {destino}")
print(f"{destino.stat().st_size/1e6:.1f} MB")
print({m: len(artefato["colunas_modelo"][m]) for m in CORTES}, "colunas por corte")

## 14. Matriz de Correlação de Pearson

Sobre o corte **m3** (fechamento, onde todas as colunas estão maduras). Serve
para achar redundância entre features e conferir a **direção** de cada sinal.

> **Sinal invertido é sintoma de vazamento, não achado contraintuitivo.** Se uma
> coluna de risco aparecer com correlação negativa contra `EVADIU` — por exemplo
> uma coluna de risco protegendo — procure o corte de data, não a explicação.

In [ ]:
M = 3
num = [c for c in feat[M].select_dtypes(include=[np.number]).columns
       if c not in ("EVADIU", "CURSOU_ALVO", "ELEGIVEL", "MES_CORTE")]
dc = feat[M][num].copy()
dc.insert(0, "EVADIU", y[M])
corr = dc.corr()

plt.figure(figsize=(13, 11))
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), cmap="coolwarm",
            vmax=0.5, vmin=-0.5, center=0, square=True, linewidths=0.4,
            annot=True, fmt=".2f", annot_kws={"size": 5},
            cbar_kws={"shrink": 0.8})
titulo = (f"Matriz de Correlação de Pearson — híbrido, corte m{M} (fechamento)" + chr(10)
          + f"({len(dc):,} alunos, {int(dc.EVADIU.sum()):,} evasores, 3 coortes)")
plt.title(titulo, fontsize=12, fontweight="bold", pad=14)
plt.xticks(rotation=45, ha="right", fontsize=6)
plt.yticks(fontsize=6)
plt.tight_layout()

pasta = RAIZ / "relatorios" / "matriz-correlacao"
if pasta.is_dir():
    saida = pasta / "matriz_correlacao_ead.png"
    plt.savefig(saida, dpi=150)
    print(f"PNG salvo em: {saida}")
plt.show()

In [ ]:
evol = {}
for m in CORTES:
    d = feat[m][[c for c in num if c in feat[m].columns]].copy()
    d["EVADIU"] = y[m]
    evol[f"m{m}"] = d.corr()["EVADIU"].drop("EVADIU")
evol = pd.DataFrame(evol).dropna()
evol["ganho"] = evol["m3"].abs() - evol["m1"].abs()

ordem = evol["m3"].abs().sort_values(ascending=False).head(14).index[::-1]
pos = np.arange(len(ordem))

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(pos + 0.20, evol.loc[ordem, "m1"], 0.38, color="#A0AEC0",
        edgecolor="black", lw=0.4, label="m1 — 30 dias de trimestre")
ax.barh(pos - 0.20, evol.loc[ordem, "m3"], 0.38, color="#1A365D",
        edgecolor="black", lw=0.4, label="m3 — trimestre fechado")
ax.axvline(0, color="gray", ls="--", lw=0.9)
ax.set_yticks(pos)
ax.set_yticklabels(ordem, fontsize=8.5)
ax.set_xlabel("Correlação de Pearson com EVADIU   —   negativo retém, positivo arrisca",
              fontsize=10, fontweight="bold")
ax.set_title("Quanto cada variável informa sobre a evasão, no início e no fim",
             fontsize=12, fontweight="bold", pad=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print("AS QUE MAIS GANHAM SINAL AO LONGO DO TRIMESTRE")
display(evol.reindex(evol["ganho"].sort_values(ascending=False).index)
        .head(8)[["m1", "m3", "ganho"]].round(3))

print("AS QUE JA INFORMAM NO m1 (nao precisam do trimestre inteiro)")
display(evol.reindex(evol["m1"].abs().sort_values(ascending=False).index)
        .head(8)[["m1", "m3", "ganho"]].round(3))

r = corr["EVADIU"].drop("EVADIU")
risco = ("AVA_DIAS_DESDE_ULT_ACESSO", "AVA_MAIOR_GAP_DIAS", "SEM_AVA")
invertidas = [c for c in risco if c in r.index and r[c] < 0]
print("colunas de RISCO com sinal invertido:", invertidas or "nenhuma -- ok")

## 15. O Teste que Decide: o modelo ganha de uma regra de uma coluna?

**Critério do projeto:** só publicar modelo que supere a regra de um critério por
margem que sobreviva ao intervalo de confiança. Foi esse teste que derrubou o
modelo da pós híbrido.

Dois cuidados, ambos aprendidos errando:

**1. O nulo tem que ser tratado como pior caso.** Nulo em
`AVA_DIAS_DESDE_ULT_ACESSO` significa "nunca acessou", não "risco zero". Mandar
os nulos para o fim da fila incapacita a regra justamente onde ela deveria
acertar.

**2. A regra tem que ser escolhida fora do conjunto de teste.** Pegar a melhor de
várias regras medindo no próprio teste favorece a regra — é seleção no teste. Aqui
ela é escolhida em **2025/24** e aplicada sem troca em 2026/21, nas mesmas
condições do modelo.

In [ ]:
ESCOLHA = "2026/21"   # ultima coorte do TREINO, nunca a de teste


def topo_k(ym, score, k=FILA):
    s = pd.Series(score).fillna(-np.inf).values
    return int(np.asarray(ym)[np.argsort(-s)[:k]].sum())


def regras_de(s):
    return {
        "AVA_DIAS_DESDE_ULT_ACESSO maior": s.AVA_DIAS_DESDE_ULT_ACESSO.fillna(1e9),
        "AVA_QTD_SESSOES menor": -s.AVA_QTD_SESSOES.fillna(0),
        "CREDITOS menor": -s.CREDITOS.fillna(0),
        "DISC_MATRICULADAS_INICIO menor": -s.DISC_MATRICULADAS_INICIO.fillna(0),
        "NOTA_MEDIA menor": -s.NOTA_MEDIA.fillna(-1),
    }


linhas_regra = []
for m in CORTES:
    d, co = feat[m], meta[m]["coorte"]
    esc = co == ESCOLHA
    k_esc = faixas(int(esc.sum()))["ALTA"]
    k_te = faixas(len(y_av[m]))["ALTA"]
    n, ev = len(y_av[m]), int(y_av[m].sum())

    placar = {nome: topo_k(y[m][esc], v, k_esc)
              for nome, v in regras_de(d[esc]).items()}
    vencedora = max(placar, key=placar.get)

    r_reg = topo_k(y_av[m], regras_de(feat_av[m])[vencedora], k_te)
    r_mod = topo_k(y_av[m], proba[m], k_te)
    trapaca = max(topo_k(y_av[m], v, k_te) for v in regras_de(feat_av[m]).values())

    linhas_regra.append({
        "Corte": f"m{m}", "Data": CORTE_DATA[m], "Fila (10%)": k_te,
        "Modelo": r_mod,
        "Regra 1 coluna": r_reg,
        f"Escolhida em {ESCOLHA}": vencedora,
        "Modelo - regra": r_mod - r_reg,
        "Acaso": round(k_te * ev / n),
        "(se escolhesse no teste)": trapaca,
    })

display(pd.DataFrame(linhas_regra))

print("Diferenca positiva = o modelo se justifica. Negativa = uma coluna faz melhor.")
print()
print()
print("ATENCAO: a fila do hibrido tem ~275 alunos, nao ~960 como a do EAD.")
print("Margens de 5 a 15 evasores aqui sao da ordem do ruido entre sementes.")
print("Este quadro e de UMA origem -- o topico 16 mede as duas.")

## 16. A Replicacao: o resultado se repete na origem seguinte?

Ate 01/10/2026 o híbrido tinha **3 coortes = 1 origem**. Com a quarta, sao **2
origens**, e pela primeira vez da para separar resultado de sorte de coorte.

```
origem 1   treina 2025/23 + 2025/24              testa 2026/21
origem 2   treina 2025/23 + 2025/24 + 2026/21    testa 2026/22
```

**As duas coortes de teste tem taxas de evasao muito diferentes** -- 13,2% contra
20,5%. Isso torna a comparacao direta enganosa, de duas formas:

1. **O acerto bruto engana.** Com mais evasor na base, encher a fila de evasor fica
   mecanicamente mais facil. Os 86,8% do m3 na coorte nova nao significam que o
   modelo ficou melhor que os 66,5% da anterior.
2. **O teto nao ajuda.** Nas duas coortes ha mais evasor do que vaga na fila, entao
   `min(evasores, fila)` e sempre o tamanho da fila -- e "aproveitamento" vira so a
   precisao com outro nome. **Nao use essa coluna para comparar origens.**

O numero comparavel e o **ganho sobre o acaso**: quantas vezes o modelo supera o que
um sorteio aleatorio pegaria naquela mesma coorte. O acaso muda junto com a taxa de
evasao, entao a razao corrige a diferenca de base.

> Erro cometido em 01/10/2026 e corrigido aqui: a primeira versao desta tabela
> chamava a precisao de "aproveitamento" e a apresentava como comparavel. Nao e.

A regra de uma coluna e escolhida sempre na **ultima coorte do treino** de cada
origem, nunca na de teste.


In [ ]:
ORIGENS = [
    {"nome": "origem 1", "treino": ["2025/23", "2025/24"],
     "teste": "2026/21", "escolha": "2025/24"},
    {"nome": "origem 2", "treino": ["2025/23", "2025/24", "2026/21"],
     "teste": "2026/22", "escolha": "2026/21"},
]


def rodar(m, treino, teste):
    """Treina nas coortes dadas e pontua a de teste. Media das sementes."""
    Xm, ym, co = X_enc[m], y[m], meta[m]["coorte"]
    tr, te = np.isin(co, treino), co == teste
    ps = []
    for s in SEMENTES:
        mod = RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                                     class_weight="balanced", n_jobs=-1,
                                     random_state=s)
        mod.fit(Xm[tr], ym[tr])
        ps.append(mod.predict_proba(Xm[te])[:, 1])
    return np.mean(ps, axis=0), te, ps


linhas_rep = []
for org in ORIGENS:
    for m in CORTES:
        pr, te, ps = rodar(m, org["treino"], org["teste"])
        yte = y[m][te]
        n, ev = len(yte), int(yte.sum())
        k = faixas(n)["ALTA"]
        teto = min(ev, k)

        esc = meta[m]["coorte"] == org["escolha"]
        k_esc = faixas(int(esc.sum()))["ALTA"]
        placar = {nome: topo_k(y[m][esc], v, k_esc)
                  for nome, v in regras_de(feat[m][esc]).items()}
        vencedora = max(placar, key=placar.get)
        r_reg = topo_k(yte, regras_de(feat[m][te])[vencedora], k)
        r_mod = topo_k(yte, pr, k)

        acaso = k * ev / n                      # o que um sorteio pegaria
        # desvio entre sementes: diz se a margem sobre a regra e sinal ou ruido
        por_semente = [topo_k(yte, p, k) for p in ps]
        desvio = round(float(np.std(por_semente)), 1)
        linhas_rep.append({
            "Origem": org["nome"], "Teste": org["teste"], "Corte": f"m{m}",
            "Evasores": ev, "Fila": k,
            "Modelo": r_mod, "Desvio": desvio, "Acerto": f"{100 * r_mod / k:.1f}%",
            "Acaso": round(acaso), "Ganho": f"{r_mod / acaso:.2f}x",
            "Recall": f"{100 * r_mod / ev:.1f}%",
            "Regra": r_reg, "Modelo - regra": r_mod - r_reg,
        })

rep = pd.DataFrame(linhas_rep)
display(rep)

print("Acerto  = dos que a fila aponta, quantos evadiram. NAO e comparavel entre")
print("          origens: sobe sozinho quando a coorte tem mais evasor.")
print("Ganho   = Modelo / Acaso. ESTE e o numero comparavel entre as origens.")
print("Recall  = dos evasores da coorte, quantos a fila alcancou. Cai quando ha")
print("          mais evasor que vaga na fila.")
print()
for m in CORTES:
    a, b = rep[(rep.Corte == f"m{m}")].iloc[0], rep[(rep.Corte == f"m{m}")].iloc[1]
    da, db = a["Modelo - regra"], b["Modelo - regra"]
    # NAO basta o sinal concordar: dois negativos "concordam" e sao reprovacao
    # nas duas. O criterio do projeto exige margem POSITIVA nas duas origens.
    if da > 0 and db > 0:
        veredito = "publica nas duas"
    elif da <= 0 and db <= 0:
        veredito = "REPROVA nas duas"
    else:
        veredito = "inconsistente -- nao da para concluir"
    print(f"  m{m}: margem sobre a regra {da:+d} -> {db:+d}   [{veredito}]")
    print(f"       ganho sobre o acaso {a['Ganho']} -> {b['Ganho']}"
          f"   |  desvio do modelo +-{a['Desvio']} / +-{b['Desvio']}")


---

# Achados específicos do híbrido

Os tópicos acima espelham o caderno do presencial. Os que seguem existem só aqui,
porque tratam de decisões que só o híbrido precisou tomar.

## 17. Estabilidade — o número se repete em outra coorte?

Deixa-uma-coorte-de-fora: cada coorte vira teste uma vez. Não é origem deslizante
(treina com futuro); o objetivo aqui é só medir oscilação.

In [ ]:
est = []
for m in CORTES:
    Xm, ym, co = X_enc[m], y[m], meta[m]["coorte"]
    for c in sorted(set(co)):
        te = co == c
        n, ev = int(te.sum()), int(ym[te].sum())
        k = faixas(n)["ALTA"]
        r = []
        for s in range(3):
            mod = RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                                         class_weight="balanced", n_jobs=-1,
                                         random_state=s)
            mod.fit(Xm[~te], ym[~te])
            r.append(topo_k(ym[te], mod.predict_proba(Xm[te])[:, 1], k))
        est.append({"Corte": f"m{m}", "Teste": c, "Alunos": n, "Evasores": ev,
                    "Sem AVA": f"{100*feat[m].loc[te, 'SEM_AVA'].mean():.0f}%",
                    "Topo 10%": f"{np.mean(r):.1f}",
                    "recall@10%": f"{100*np.mean(r)/ev:.1f}%"})

display(pd.DataFrame(est))
print("A 2025/23 cai junto com a cobertura do AVA -- entrada faltando, nao modelo")
print("ruim. Note que ela so e ruim no m1: conforme a cobertura melhora, ela")
print("recupera e no m3 e a melhor das tres.")

## 18. Quem escapa — e por que coluna nova não resolve

Se o evasor que escapa for indistinguível de quem permaneceu, **nenhuma coluna
acadêmica nova vai separá-los** — e procurar mais seria desperdício. Foi o que
aconteceu na pós híbrido e o que motivou o bloco financeiro aqui.

In [ ]:
m = 1
sub = feat_av[m].copy()
sub["P"] = proba[m]
sub["EVADIU"] = y_av[m]
k = faixas(len(sub))["ALTA"]
sub["NA_FILA"] = sub.P.rank(ascending=False, method="first") <= k

ev = sub[sub.EVADIU == 1]
cap, esc = ev[ev.NA_FILA], ev[~ev.NA_FILA]
fic = sub[(sub.EVADIU == 0) & (~sub.NA_FILA)]
print(f"corte m{m}, teste {TESTE}: {len(ev):,} evasores")
print(f"   capturados nos 10%: {len(cap):,} ({100*len(cap)/len(ev):.1f}%)")
print(f"   escapados         : {len(esc):,} ({100*len(esc)/len(ev):.1f}%)")
print()

cols = [c for c in ["AVA_QTD_SESSOES", "AVA_DIAS_ATIVOS", "AVA_DIAS_DESDE_ULT_ACESSO",
                    "CREDITOS", "DISC_MATRICULADAS_INICIO", "NOTA_MEDIA",
                    "TAXA_APROVACAO",
                    "IDADE_INGRESSO", "ANOS_DE_CASA"] if c in sub.columns]
comp = pd.DataFrame({"Capturado": cap[cols].mean(), "ESCAPADO": esc[cols].mean(),
                     "Permaneceu": fic[cols].mean()}).round(2)
comp["Escapado parece"] = np.where(
    (comp.ESCAPADO - comp.Capturado).abs() < (comp.ESCAPADO - comp.Permaneceu).abs(),
    "capturado", "quem FICOU")
display(comp)

## 19. Auditoria de Viés por Sexo

`SEXO` não entra no `X`. Mas o desvio pode entrar por **proxy** — no híbrido ele
sobreviveu à remoção da coluna porque `CURSO` carregava a informação.

O que se mede é **recall por grupo**: dos evasores de cada sexo, quantos a fila
alcança. Não é taxa de alerta.

> Erro já cometido no projeto: usar `pred.sum()` no numerador, o que inclui falso
> positivo e reportou "recall de 177,5% nas mulheres".

In [ ]:
linhas_vies = []
for m in CORTES:
    co = meta[m]["coorte"]
    te = co == TESTE
    k = faixas(int(te.sum()))["ALTA"]
    sx = pd.Series(meta[m]["sexo"][te]).astype(str).str.upper().str[0].values
    yy = y[m][te]
    na_fila = (pd.Series(proba[m]).rank(ascending=False, method="first") <= k).values

    linha, rec = {"Corte": f"m{m}"}, {}
    for g, rotulo in (("F", "Mulheres"), ("M", "Homens")):
        sel = sx == g
        evg = int(yy[sel].sum())
        pego = int((yy[sel] & na_fila[sel]).sum())
        rec[g] = 100 * pego / evg if evg else float("nan")
        linha[rotulo] = f"{rec[g]:.1f}%  ({pego} de {evg})"
    linha["Gap"] = f"{abs(rec['F'] - rec['M']):.1f} pp"
    linhas_vies.append(linha)

display(pd.DataFrame(linhas_vies))
print(f"Fila de {k:,} alunos (10% da base), coorte {TESTE}.")
print("Gap acima de ~5 pp merece investigar proxy -- comecar por CURSO.")

## 20. O que fica em aberto

1. **Uma origem só.** A validação honesta tem um único teste (2026/21). Em
   01/10/2026 a coorte 2026/22 fecha e passa a haver duas — reextrair e refazer.
2. **O m2 não publica.** Fica fora do artefato de produção até que alguém o supere.
3. **Dependência do AVA.** Onde a cobertura do log falha, a fila degrada muito e
   sem aviso. Vale medir o modelo sem nenhuma coluna de AVA para saber o risco.
4. **Blocos ainda ausentes:** requerimentos administrativos (bloco 9 do presencial)
   e bolsa (bloco 11). Requerimento de trancamento tem data e é sinal precoce.
5. **Vazamento herdado no presencial:** o bloco financeiro de lá lê baixa
   posterior ao corte. Corrigir antes de reativar.
6. **`pop` não filtra por data** nas três modalidades — menos de 3% dos alunos,
   registrado no CLAUDE.md.